# 📗 LangChain 기본 구조 — Runnable·Memory·대화 워크플로우

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞 시간에 우리는 **프롬프트 → 모델 → 파서** 를 파이프 `|` 로 잇는 체인을 만들었습니다. 이번 시간에는 이 부품들의 **공통 규약**인 **Runnable** 을 이해하고, 내 파이썬 함수를 부품으로 만드는 방법, 여러 일을 **동시에** 처리하는 방법, 그리고 모델이 앞 대화를 **기억**하게 하는 **대화 워크플로우**를 만듭니다. 마지막에는 잘라 낸 대화까지 되살리는 **장기 기억**을 붙여 마무리합니다.

## ⏪ 복습 — 지난 시간: 모델·프롬프트·체인

- **모델**: `ChatOpenAI(...)` 로 만들고 `.invoke(...)` 로 답을 받았습니다(응답 텍스트는 `.text`).
- **프롬프트 템플릿**: `ChatPromptTemplate.from_messages([...])` 로 `{변수}` 를 끼운 프롬프트를 만들었습니다.
- **출력 파서**: `StrOutputParser()` 로 응답에서 문자열만 뽑았습니다.
- **체인(LCEL)**: `프롬프트 | 모델 | 파서` 를 잇고 `.invoke`·`.batch` 로 실행했습니다.

오늘은 여기에 **함수 부품**·**병렬**·**기억**을 더합니다.

### 📚 공식 문서 — 오늘 배우는 것들

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| 부품을 잇는 구조(Runnable·파이프) | [Component architecture](https://docs.langchain.com/oss/python/langchain/component-architecture) · [langchain_core.runnables](https://reference.langchain.com/python/langchain-core/runnables) |
| `stream` 으로 조각 받기 | [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) |
| 대화 기록(단기 기억) | [Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory) |
| 오래 쓸 사실(장기 기억) | [Long-term memory](https://docs.langchain.com/oss/python/langchain/long-term-memory) |
| 기억을 왜 나누나 | [Memory overview](https://docs.langchain.com/oss/python/concepts/memory) |

> 공식 문서는 **에이전트(다음 단원)** 를 중심으로 쓰여 있어, 오늘 배우는 내용이 그 안에 섞여 나옵니다. 지금은 위 표의 링크만 따라가면 충분하고, 나머지는 다음 단원부터 하나씩 만나게 됩니다.

**오늘의 목표**

- [ ] **Runnable** — 모든 부품의 공통 규약(`invoke`·`batch`·`stream`)을 이해하고 스트리밍을 본다.
- [ ] **RunnableLambda** — 내 파이썬 함수를 체인의 부품으로 끼운다.
- [ ] **RunnableParallel** — 하나의 입력을 나눠 여러 일을 **동시에** 처리한다(요약+감정).
- [ ] **대화 기록(Memory)** — 모델은 기억이 없음을 실연하고, 기록 리스트 + `MessagesPlaceholder` 로 기억을 준다.
- [ ] **기록 관리** — 대화를 누적하고 최근 N턴만 남기는 트리밍을 만든다.
- [ ] **장기 기억** — 잘라 낸 사실을 벡터 저장소에 남기고 질문과 가까운 것만 회상한다.
- [ ] **자동 판단** — 무엇을 기억할지 모델에게 물어보고, 저장 여부는 코드가 정한다.
- [ ] **멀티턴 상담 워크플로우** — 역할 프롬프트 + 장기 기억 + 단기 기록 + 체인을 하나로 종합한다.

아래 준비 셀을 먼저 실행하세요(앞 시간과 같은 `.env` 의 `OPENAI_API_KEY` 를 씁니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이번 시간 공통 부품 — 앞 시간에 배운 모델·파서
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
parser = StrOutputParser()

---
# 1. Runnable — 모든 부품의 공통 규약

## 왜 중요할까요?
지금까지 만든 **모델·프롬프트·파서·체인** 은 겉모습은 달라도 **부르는 방법이 똑같습니다**. 전부 `.invoke(...)`·`.batch([...])`·`.stream(...)` 을 가집니다. 이 공통 규약을 **Runnable** 이라고 부릅니다. 그래서 파이프 `|` 로 자유롭게 이어 붙일 수 있는 것입니다.

## 문법 — 세 가지 실행 방법
- **`invoke(입력)`**: 하나를 처리하고 결과 하나를 받습니다.
- **`batch([입력들])`**: 여러 개를 한꺼번에 처리합니다. (동시에 처리한다, 동시 처리 개수도 제한도 가능)
- **`stream(입력)`**: 답을 **조각(청크) 단위로 흘려** 받습니다. 긴 답을 기다리지 않고 **먼저 나오는 부분부터** 보여 줄 때 씁니다(챗봇의 타이핑 효과).

<img src="images/runnable_three.png" width="760">

> 앞의 둘(`invoke`·`batch`)은 지난 시간에 이미 써 봤습니다. **이 절에서 새로운 것은 `stream` 하나**입니다 — 다만 셋을 **같은 체인 하나**에 나란히 걸어 보면, 이것이 부품마다 따로 있는 기능이 아니라 **모든 부품이 공유하는 규약**이라는 점이 눈에 들어옵니다.

먼저 이 절에서 계속 쓸 **상담 체인 하나**를 만들어 둡니다. 지난 시간에 배운 `프롬프트 | 모델 | 파서` 구조 그대로입니다. 만들어 두고, 아래에서 **같은 체인을 세 가지 방법으로** 실행해 봅니다.

In [ ]:
# 이 절 내내 재사용할 체인 — 지난 시간의 '프롬프트 | 모델 | 파서' 구조 그대로입니다.
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 상담원이다. 존댓말로 간결하게 답한다.'),
    ('human', '{input}'),
])
simple_chain = chat_prompt | model | parser

# 확인 포인트: 부품 셋을 이어 붙인 결과도 결국 '부품 하나'이고,
#   그 하나가 invoke·batch·stream 을 전부 갖고 있다는 것 — 이것이 Runnable 규약입니다.
print('체인의 종류    :', type(simple_chain).__name__)
print('invoke 가 있나 :', hasattr(simple_chain, 'invoke'))
print('batch 가 있나  :', hasattr(simple_chain, 'batch'))
print('stream 이 있나 :', hasattr(simple_chain, 'stream'))

### 1) `invoke` — 하나 넣고 하나 받기

가장 기본입니다. 지금까지 계속 써 온 방법이죠. 답이 **다 만들어진 뒤에** 한꺼번에 돌아옵니다.

In [ ]:
# invoke: 입력 하나 → 결과 하나. 답이 완성될 때까지 기다렸다가 받습니다.
one = simple_chain.invoke({'input': '고양이 스크래처는 어떤 재질이 좋아? 한 문장으로 답해줘.'})
print('결과의 종류:', type(one).__name__)   # 파서가 끝에 있어 str
print(one)

### 2) `batch` — 여러 개를 한꺼번에

입력을 **리스트**로 주면 결과도 **리스트**로 돌아옵니다. 질문이 여러 개일 때 하나씩 `invoke` 하는 것보다 간결합니다.

In [ ]:
# batch: 입력 리스트 → 결과 리스트. 순서는 넣은 순서 그대로 유지됩니다.
questions = [
    {'input': '강아지 방석은 얼마나 자주 세탁해? 한 문장으로.'},
    {'input': '산책 리드줄 길이는 어느 정도가 좋아? 한 문장으로.'},
]
answers = simple_chain.batch(questions)

# 확인 포인트: 넣은 개수와 받은 개수가 같고, 순서가 어긋나지 않았는지.
print('넣은 개수:', len(questions), '/ 받은 개수:', len(answers))
for q, a in zip(questions, answers):
    print('-', q['input'])
    print('  →', a)

### 3) `stream` — 조각으로 흘려 받기

답 전체를 기다리지 않고 **먼저 만들어진 부분부터** 받습니다. 챗봇에서 글자가 타이핑되듯 나오는 것이 이것입니다.

In [ ]:
# stream: 답을 조각(청크)으로 흘려 받습니다.
#   end='' 는 조각마다 줄바꿈하지 않고 이어 붙여 출력하려는 것입니다.
print('스트리밍 출력: ', end='')
pieces = []
for chunk in simple_chain.stream({'input': '반려견과 첫 산책을 나갈 때 준비물을 세 가지만 알려줘.'}):
    print(chunk, end='')
    pieces.append(chunk)
print()

# 확인 포인트: 조각을 이어 붙이면 결국 하나의 완성된 답이 된다는 것.
print('조각 개수      :', len(pieces))
print('이어 붙인 길이 :', len(''.join(pieces)))

> 답이 **여러 조각**으로 나뉘어 흘러나왔습니다. 조각을 나누는 기준은 모델이 정하므로 **조각 개수는 실행할 때마다 달라집니다** — 그래서 스트리밍 결과는 개수가 아니라 **이어 붙인 문자열**로 다룹니다.

### ✅ 바로 확인 퀴즈

**1.** 모델·프롬프트·파서·체인이 모두 공유하는 공통 규약의 이름은?

<details><summary>정답 보기</summary>

**Runnable** 입니다. 그래서 모두 `invoke`·`batch`·`stream` 을 갖고, 파이프 `|` 로 이을 수 있습니다.

</details>

**2.** 답을 조각 단위로 흘려 받는 실행 방법은?

<details><summary>정답 보기</summary>

**`stream(...)`** 입니다. 조각을 이어 붙이면 전체 답이 됩니다.

</details>

---
# 2. RunnableLambda — 내 함수를 부품으로

## 왜 필요할까요?
체인 중간에 **내가 만든 파이썬 함수**를 끼우고 싶을 때가 있습니다(입력 정리, 값 가공 등). 그냥 함수는 파이프 `|` 로 이을 수 없지만, **`RunnableLambda(함수)`** 로 감싸면 **체인 부품**이 됩니다.

## 문법 — `RunnableLambda(함수)`
함수는 입력 하나를 받아 결과 하나를 돌려주면 됩니다. 아래는 사용자 입력의 **앞뒤·연속 공백을 정리**하는 함수를 부품으로 만들어, 프롬프트 앞에 붙인 예입니다.

In [ ]:
from langchain_core.runnables import RunnableLambda

def clean_text(text):
    """앞뒤 공백을 없애고 중간 연속 공백을 한 칸으로 줄인다."""
    return ' '.join(text.split())

# 함수를 부품으로 감싼다
clean_step = RunnableLambda(clean_text)
print('정리 결과:', clean_step.invoke('  강아지   방석  추천 '))

이 정리 함수를 프롬프트 앞에 끼워, **입력을 먼저 다듬고** 모델에 넣는 체인을 만듭니다. 사용자 입력은 `{input}` 변수로 들어가므로, 정리한 문자열을 그 변수에 담아 넘깁니다.

In [ ]:
# clean_text 로 정리한 문자열을 {input} 에 담아 넘기는 체인
ask_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 상담원이다. 존댓말로 간결하게 답한다.'),
    ('human', '{input}'),
])
clean_chain = RunnableLambda(lambda q: {'input': clean_text(q)}) | ask_prompt | model | parser
print(clean_chain.invoke('  캣타워는   어떤 걸 골라야  하나요? '))

### 🖐️ 함께 따라하기 — 답 끝에 안내 문구를 붙이는 부품

상담 답변 끝에는 보통 **안내 문구**가 따라붙습니다. 문자열을 받아 그 끝에 줄바꿈과 함께 `'※ 더 궁금한 점은 상담 채팅으로 문의해 주세요.'` 를 덧붙여 돌려주는 함수를 만들고, `RunnableLambda` 로 감싸 앞 `clean_chain` **뒤**에 이어 보세요(즉 `clean_chain | RunnableLambda(내함수)` 형태). 아무 질문이나 `invoke` 해 답 끝에 안내가 붙는지 확인하세요.

**확인 기준**: 출력의 마지막 줄이 그 안내 문구다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 문자열을 받아 끝에 줄바꿈 + '※ 더 궁금한 점은 상담 채팅으로 문의해 주세요.' 를 붙여 돌려주는 함수를 만든다
# 2) RunnableLambda 로 감싸 clean_chain 뒤에 파이프로 잇는다
# 3) 아무 질문이나 invoke 해 답 끝에 안내 문구가 붙는지 확인한다

## 이어서 — 체인과 체인을 한 체인으로

지난 시간 마지막에 **2단 흐름**(홍보 문구 → 한 줄 광고)을 만들었습니다. 그때는 앞 체인을 `invoke` 해서 결과를 변수에 담고, 그 변수를 뒤 체인의 `invoke` 에 손으로 넣었습니다. 두 번 부른 셈입니다.

왜 그냥 `앞체인 | 뒤체인` 으로 잇지 못했을까요? **모양이 안 맞기 때문**입니다 — 앞 체인은 파서로 끝나 **문자열**을 내는데, 뒤 프롬프트는 `{draft}` 를 채울 **딕셔너리**를 원합니다. 이 사이에 모양을 바꿔 주는 부품 하나만 끼우면 **두 체인이 하나가 됩니다** — 그 부품이 방금 배운 `RunnableLambda` 입니다.

**(1단계) 이을 두 체인을 먼저 만듭니다.** 지난 시간에 쓰던 것과 같은 두 체인입니다.

In [ ]:
# 앞 체인: 상품 정보 -> 홍보 문구(긴 문장)
promo_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 홍보 문구를 쓰는 카피라이터다.'),
    ('human', '다음 상품의 홍보 문구를 한 문장으로 써줘. 이름: {name}, 특징: {keywords}'),
])
promo_chain = promo_prompt | model | parser

# 뒤 체인: 긴 문구 -> 짧은 한 줄 광고. 채울 변수 이름이 {draft} 라는 점을 기억해 둡니다.
shorten_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 긴 문구를 짧고 강한 한 줄 광고로 다듬는 편집자다.'),
    ('human', '다음 문구를 12자 안팎의 한 줄 광고로 줄여줘:\n{draft}'),
])
shorten_chain = shorten_prompt | model | parser
print('두 체인 준비 완료')

**(2단계) 왜 바로 이을 수 없는지 눈으로 확인합니다.** 앞 체인을 한 번 실행해 **무엇이 나오는지** 보세요.

In [ ]:
# 앞 체인의 출력이 어떤 '모양'인지 확인합니다.
draft_text = promo_chain.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'})
print('앞 체인 출력의 종류:', type(draft_text).__name__)   # str — 그냥 문자열
print(draft_text)

# 그런데 뒤 체인은 {draft} 를 채울 '딕셔너리'를 원합니다 -> 모양이 안 맞아 바로 이을 수 없습니다.
print('뒤 체인이 원하는 모양 예시:', {'draft': '(여기에 문자열)'})

**(3단계) 모양을 바꿔 주는 다리 부품을 만들고, 그것만 따로 실행해 봅니다.** 체인에 끼우기 전에 이 부품 하나가 제대로 도는지 먼저 확인하는 것이 디버깅에 좋습니다.

In [ ]:
# 문자열 하나를 받아 {'draft': 그 문자열} 로 바꿔 주기만 하는 작은 부품
to_draft = RunnableLambda(lambda text: {'draft': text})

# 부품 하나만 단독 실행 — 모델을 부르지 않으므로 즉시, 항상 같은 결과가 나옵니다.
print(to_draft.invoke('테스트 문구'))   # {'draft': '테스트 문구'}

**(4단계) 세 부품을 한 줄로 잇습니다.** 이제 모양이 맞으므로 `앞체인 | 다리 | 뒤체인` 이 하나의 체인이 됩니다.

In [ ]:
# 앞 체인(문자열) -> 다리(딕셔너리로 변환) -> 뒤 체인 : 모양이 맞아 하나로 이어집니다.
two_step_chain = promo_chain | to_draft | shorten_chain

# 확인 포인트: 2단계에서 invoke 를 두 번 불렀지만, 여기서는 '한 번'이면 끝난다는 것.
print(two_step_chain.invoke({'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'}))

> 지난 시간에는 `invoke` 를 두 번 불렀지만, 지금은 **한 번**입니다. 흐름 전체가 **부품 하나**가 되었기 때문입니다. 이렇게 단계를 선언적으로 이어 붙이는 것이 LCEL 로 **순차 흐름**을 설계하는 방법입니다.

**부품 하나가 되면 무엇이 좋을까요?** 1절의 **Runnable 규약**이 이 합쳐진 체인에도 그대로 적용됩니다 — `invoke` 뿐 아니라 `batch`·`stream` 도 쓸 수 있다는 뜻입니다. 상품 두 개를 **한꺼번에** 돌려 확인해 봅시다. (1절에서 손으로 잇던 방식이었다면 상품마다 `invoke` 를 두 번씩, 네 번 불러야 했습니다.)

In [ ]:
# 합쳐진 체인도 결국 부품 하나 -> batch 를 그대로 쓸 수 있습니다.
#   상품 2개 x (홍보 문구 -> 한 줄 광고) 2단계 = 모델 호출 4번을 batch 가 알아서 돌립니다.
two_step_inputs = [
    {'name': '포근 강아지 방석', 'keywords': '메모리폼, 미끄럼 방지 바닥, 세탁 가능'},
    {'name': '발수 강아지 우비', 'keywords': '방수 원단, 배 부분 밴드, 형광 로고'},
]
two_step_results = two_step_chain.batch(two_step_inputs)

# 확인 포인트: 넣은 개수와 받은 개수가 같고, 상품과 결과의 짝이 맞는지.
for item, out in zip(two_step_inputs, two_step_results):
    print('-', item['name'], '→', out)

> 두 상품의 2단 흐름이 **`batch` 한 번**으로 끝났습니다. 체인을 하나로 이어 두면 이렇게 **`invoke`·`batch`·`stream` 을 전부** 쓸 수 있습니다 — 손으로 이었다면 상품마다 `invoke` 를 두 번씩 불러야 했겠죠.

**단, `stream` 은 한 가지를 알아 두세요.** 중간의 다리 부품(`to_draft`)은 앞 체인의 **문자열 전체**를 받아야 일할 수 있습니다. 그래서 이 체인을 `stream` 하면 **앞 단계가 끝날 때까지 기다렸다가** 뒤 단계부터 조각이 흘러나옵니다(에러는 나지 않고 결과도 같습니다). 챗봇에서 **첫 글자가 빨리 뜨는** 효과를 노린다면 체인의 **마지막 단계가 모델**이어야 합니다.

### 🖐️ 함께 따라하기 — 다리를 직접 놓아 두 체인 잇기

이번엔 **다리 부품을 직접 만들어** 두 체인을 하나로 이어 보세요. 앞 `promo_chain` 뒤에 **"홍보 문구를 반말 한 줄 후기처럼 바꾸는"** 체인을 잇습니다.

1. `{draft}` 를 받아 **반말 한 줄 후기**로 바꾸라는 프롬프트로 `casual_chain` 을 만든다(`프롬프트 | model | parser`).
2. 앞 체인이 내는 **문자열**을 뒤 체인이 원하는 **딕셔너리**로 바꾸는 다리 부품을 `RunnableLambda` 로 만든다.
3. `promo_chain | 다리 | casual_chain` 을 한 체인으로 잇고, 상품 **이름="발수 강아지 우비", 특징="방수 원단, 배 부분 밴드, 형광 로고"** 로 `invoke` 를 **한 번만** 불러 결과를 출력한다.

**확인 기준**: `invoke` 를 한 번만 부르는데 결과가 반말 한 줄로 나온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) {draft} 를 받아 반말 한 줄 후기로 바꾸는 casual_chain 을 만든다
# 2) 문자열을 {'draft': 문자열} 로 바꾸는 다리 부품을 RunnableLambda 로 만든다
# 3) promo_chain | 다리 | casual_chain 을 이어 invoke 를 한 번만 부른다
# 4) 상품은 이름 '발수 강아지 우비', 특징 '방수 원단, 배 부분 밴드, 형광 로고'

### ✅ 바로 확인 퀴즈

**1.** 내 파이썬 함수를 체인 부품으로 만들려면 무엇으로 감싸나요?

<details><summary>정답 보기</summary>

**`RunnableLambda(함수)`** 로 감쌉니다. 그러면 파이프 `|` 로 다른 부품과 이을 수 있습니다.

</details>

**2.** 문자열을 내는 체인 뒤에 `{draft}` 를 받는 체인을 바로 이으면 왜 안 되나요? 어떻게 해결했나요?

<details><summary>정답 보기</summary>

앞 체인의 출력은 **문자열**인데 뒤 프롬프트는 변수를 채울 **딕셔너리**를 받기 때문입니다. 사이에 `RunnableLambda(lambda text: {'draft': text})` 를 끼워 **모양을 바꿔** 이었습니다.

</details>

---
# 3. RunnableParallel — 하나의 입력으로 여러 일을 동시에

## 왜 필요할까요?
고객 리뷰 하나를 받아 **요약도 하고, 감정도 판단**하고 싶다면, 두 체인을 각각 부를 수도 있지만 **동시에** 실행하면 깔끔합니다. **`RunnableParallel`** 은 같은 입력을 **여러 부품에 나눠 넣고** 결과를 **딕셔너리 하나**로 모아 줍니다.

## 문법 — `RunnableParallel(key=부품, ...)`
- 각 키에 부품(체인)을 두면, 같은 입력이 **모든 부품에 동시에** 들어갑니다.
- 결과는 `{'키1': 결과1, '키2': 결과2}` 형태로 나옵니다.
- **`RunnablePassthrough`** 는 입력을 **그대로 통과**시키는 부품입니다(원본도 함께 남기고 싶을 때).

<img src="images/runnable_parallel.png" width="760">

**(1단계) 묶을 부품들을 먼저 각각 만들고, 하나씩 실행해 확인합니다.** 병렬로 묶은 뒤에 문제가 생기면 어느 쪽이 잘못됐는지 찾기 어렵기 때문에, 부품 단위로 먼저 확인하는 습관이 좋습니다.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 두 체인 모두 같은 이름의 변수 {input} 을 쓴다 — 그래야 같은 입력을 나눠 받을 수 있습니다.
summary_chain = (
    ChatPromptTemplate.from_messages([('human', '다음 리뷰를 한 문장으로 요약해줘:\n{input}')])
    | model | parser)

sentiment_chain = (
    ChatPromptTemplate.from_messages([('human', '다음 리뷰의 감정을 긍정/부정/중립 중 하나로만 답해줘:\n{input}')])
    | model | parser)

review = '주문한 방석이 다음 날 바로 왔어요. 두께가 도톰하고 강아지가 아주 좋아합니다. 다만 커버 지퍼가 좀 뻑뻑한 게 아쉬웠어요.'

# 아직 병렬이 아닙니다 — 하나씩 불러 각 체인이 제대로 도는지부터 봅니다.
print('요약:', summary_chain.invoke(review))
print('감정:', sentiment_chain.invoke(review))

**(2단계) 두 부품을 `RunnableParallel` 로 묶습니다.** 같은 입력이 두 체인에 **동시에** 들어가고, 결과는 **딕셔너리 하나**로 모입니다. `original` 갈래에는 `RunnablePassthrough` 를 두어 **원본도 함께** 남깁니다.

In [ ]:
# 각 키에 부품을 하나씩 — 키 이름이 그대로 결과 딕셔너리의 키가 됩니다.
analyze = RunnableParallel(
    summary=summary_chain,
    sentiment=sentiment_chain,
    original=RunnablePassthrough(),   # 입력을 가공 없이 그대로 통과시킨다
)

# 확인 포인트: invoke 한 번으로 세 갈래 결과가 한 딕셔너리에 모인다는 것.
result = analyze.invoke(review)
print('결과의 키:', list(result.keys()))
print('요약  :', result['summary'])
print('감정  :', result['sentiment'])
print('원본  :', result['original'][:20], '...')
print('원본이 손대지 않고 그대로인가?', result['original'] == review)

In [ ]:
result

> 결과가 **딕셔너리 하나**로 모였습니다(`summary`·`sentiment`·`original`). 서로 다른 분석을 한 번에 받아 뒷단계에서 함께 쓰기 좋습니다.

### 🖐️ 함께 따라하기 — 다른 리뷰를 동시에 분석

위 `analyze` 부품을 다른 리뷰에 그대로 써 보세요. 아래 리뷰를 `analyze.invoke(...)` 에 넣어 `summary`·`sentiment` 를 각각 출력하세요.

> 리뷰: "배송은 빨랐지만 스크래처 골판지 밀도가 생각보다 낮아서 금방 부서질까 걱정돼요."

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 위 analyze 부품에 아래 리뷰를 invoke 한다
# 2) 결과 딕셔너리에서 'summary' 와 'sentiment' 를 꺼내 출력한다
review2 = '배송은 빨랐지만 스크래처 골판지 밀도가 생각보다 낮아서 금방 부서질까 걱정돼요.'

### ✅ 바로 확인 퀴즈

**1.** `RunnableParallel` 의 결과는 어떤 형태로 나오나요?

<details><summary>정답 보기</summary>

**딕셔너리**입니다. 각 키에 그 부품의 결과가 담깁니다(예: `{'summary': ..., 'sentiment': ...}`).

</details>

**2.** 입력을 그대로 통과시키는 부품은?

<details><summary>정답 보기</summary>

**`RunnablePassthrough`** 입니다. 원본 입력을 가공 없이 다음으로 넘깁니다.

</details>

---
# 4. 대화 기록(Memory) — 모델에게 기억을 주기

## 왜 필요할까요?
언어 모델은 **한 번의 호출**만 기억합니다. 방금 한 말을 다음 호출에서는 **모릅니다**. 먼저 이 사실을 직접 확인해 봅니다.

**(1단계) 먼저 모델에게 정보를 알려 줍니다.** 모델이 잘 알아들었는지 답을 확인하세요.

In [ ]:
# 강아지의 이름·품종·나이를 알려 줍니다. 이 호출에서는 모델이 분명히 알아듣습니다.
told = model.invoke('우리 강아지 이름은 초코야. 소형견이고 두 살이야.')
print(told.text)

**(2단계) 이제 같은 모델에게 따로 물어봅니다.** 방금 알려 준 내용을 기억하고 있을까요?

In [ ]:
# 앞 호출과 '별개의' 호출입니다 — 방금 한 대화를 함께 넣어 주지 않았습니다.
forgot = model.invoke('초코한테 맞는 방석을 추천해줘.')
print(forgot.text)

# 확인 포인트: 초코가 '두 살 소형견'이라는 것을 모른 채 일반론으로 답합니다.
#   모델이 깜빡한 것이 아니라, 우리가 앞 대화를 넘겨 주지 않았기 때문입니다.

## 해결 — 대화 기록을 프롬프트에 끼운다
모델이 기억하게 하려면, **지금까지의 대화**를 매 호출마다 프롬프트에 **함께** 넣어 주면 됩니다. 이를 위해 프롬프트 중간에 **대화 기록이 들어갈 자리**를 비워 두는 부품이 **`MessagesPlaceholder`** 입니다.

## 문법 — `MessagesPlaceholder('history')`
- 프롬프트를 `[('system', 역할), MessagesPlaceholder('history'), ('human', '{input}')]` 로 구성합니다.
- 대화 기록은 **`(역할, 내용)` 튜플의 리스트**로 만들어 `invoke({'history': 기록, 'input': 질문})` 에 넘깁니다.
- 역할은 사용자면 `'user'`, 모델이면 `'assistant'` 를 씁니다.

<img src="images/memory_prompt.png" width="820">

**(1단계) 기록 자리를 둔 프롬프트와 체인을 만듭니다.**

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

# 역할(system) -> [기록이 들어갈 자리] -> 이번 질문(human) 순서로 구성합니다.
#   MessagesPlaceholder 는 '여기에 지난 대화를 끼워 넣겠다'는 빈자리 표시입니다.
memory_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 상담원이다. 존댓말로 간결하게 답한다.'),
    MessagesPlaceholder('history'),
    ('human', '{input}'),
])
memory_chain = memory_prompt | model | parser
print('기록 자리를 둔 체인 준비 완료')

**(2단계) 지난 대화를 기록으로 만들고, 프롬프트가 실제로 어떻게 채워지는지 먼저 눈으로 봅니다.** 여기서는 **모델을 부르지 않습니다** — 프롬프트만 완성해 확인하는 단계입니다.

In [ ]:
# 기록은 (역할, 내용) 튜플의 리스트입니다. 사용자는 'user', 모델은 'assistant'.
history = [
    ('user', '우리 강아지 이름은 초코야. 소형견이고 두 살이야.'),
    ('assistant', '네, 초코는 두 살 소형견이군요! 무엇을 도와드릴까요?'),
]

# 확인 포인트: 빈자리에 기록 두 줄이 실제로 끼워져, 모델에게는 '네 개의 메시지'가 간다는 것.
for m in memory_prompt.format_messages(history=history, input='초코한테 맞는 방석을 추천해줘.'):
    print(f'{type(m).__name__:15s}: {m.content}')

**(3단계) 이제 같은 질문을 기록과 함께 모델에 넣습니다.** 2단계에서 본 그 프롬프트가 그대로 전달됩니다.

In [ ]:
# 아까 '기억 못 했던' 바로 그 질문입니다 — 이번에는 history 를 함께 넘깁니다.
answer = memory_chain.invoke({'history': history, 'input': '초코한테 맞는 방석을 추천해줘.'})
print(answer)

# 확인 포인트: 이번에는 '초코'가 두 살 소형견임을 알고 답합니다.
#   모델에 기억 기능이 생긴 것이 아니라, 우리가 지난 대화를 함께 넣어 준 것뿐입니다.

### 🖐️ 함께 따라하기 — 한 턴 더 이어 묻기

방금 답변까지 기록에 넣고 이어서 물어보세요. 위 `history` 에 방금 질문("초코한테 맞는 방석을 추천해줘.")과 그 답이 담긴 변수 `answer` 를 한 턴으로 덧붙인 뒤, `memory_chain` 에 새 질문 "그럼 초코 간식도 하나 추천해줘." 를 넣어 출력하세요. 모델이 여전히 '초코'를 기억하는지 확인합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) history 뒤에 ('user', '초코한테 맞는 방석을 추천해줘.') 와 ('assistant', answer) 를 덧붙여 history2 를 만든다
# 2) memory_chain 에 {'history': history2, 'input': '그럼 초코 간식도 하나 추천해줘.'} 를 invoke 한다
# 3) 결과를 출력한다 (모델이 '초코'를 기억하는지 확인)

### ✅ 바로 확인 퀴즈

**1.** 언어 모델이 앞 대화를 기억하게 하려면 무엇을 해야 하나요?

<details><summary>정답 보기</summary>

**지금까지의 대화 기록을 매 호출 프롬프트에 함께 넣습니다.** 프롬프트에 `MessagesPlaceholder` 로 기록 자리를 두고, `(역할, 내용)` 튜플 리스트를 넘깁니다.

</details>

**2.** 대화 기록이 들어갈 자리를 비워 두는 프롬프트 부품은?

<details><summary>정답 보기</summary>

**`MessagesPlaceholder('history')`** 입니다.

</details>

---
# 5. 기록 관리 — 누적과 트리밍

## 왜 필요할까요?
대화가 이어지면 기록에 **질문과 답을 계속 쌓아야** 합니다. 그런데 무한정 쌓으면 프롬프트가 너무 길어져 비용·속도에 불리합니다. 그래서 **최근 N턴만 남기는 트리밍**을 함께 씁니다. 이 두 가지는 **순수 파이썬 함수**로 만들 수 있습니다(모델과 무관 — 그래서 결과가 항상 일정).

## 문법 — 누적 함수와 트리밍 함수
- **누적**: 기록 리스트에 `('user', 질문)` 과 `('assistant', 답)` 을 차례로 덧붙입니다.
- **트리밍**: 한 **턴**은 사용자+모델 두 줄이므로, 최근 N턴은 리스트의 **뒤에서 2×N개**를 남깁니다.

**(1단계) 누적 함수를 만들고 바로 확인합니다.** 모델을 부르지 않으므로 결과가 항상 같습니다 — 이런 함수는 만들자마자 한 줄로 검증할 수 있습니다.

In [ ]:
def add_turn(history, user_text, ai_text):
    """대화 기록에 사용자 질문과 모델 답을 한 턴으로 덧붙인 새 리스트를 돌려준다."""
    # 원본 리스트를 고치지 않고 '+' 로 새 리스트를 만든다 — 원본이 바뀌는 사고를 막는다.
    return history + [('user', user_text), ('assistant', ai_text)]

# 확인 포인트: 빈 기록에 한 턴을 넣으면 '두 줄'이 된다.
print(add_turn([], '안녕하세요', '안녕하세요! 무엇을 도와드릴까요?'))



hist = add_turn([], '안녕하세요', '안녕하세요! 무엇을 도와드릴까요?')
print(add_turn(hist, '오늘 점심메뉴 추천해주세요.', '김밥 추천드려요!'))

**(2단계) 트리밍 함수를 만들고 바로 확인합니다.** 한 턴이 두 줄이므로 최근 N턴은 뒤에서 `2*N` 줄입니다.

In [ ]:
def keep_recent(history, n_turns):
    """대화 기록에서 최근 n_turns 턴(2×n_turns 줄)만 남긴다."""
    # 음수 인덱스 슬라이싱 — 리스트 '뒤에서' 2*n_turns 개를 남긴다.
    return history[-2 * n_turns:]

# 확인 포인트: 네 줄(2턴)에서 최근 1턴만 남기면 뒤의 두 줄이 남는다.
sample = [('user', 'a'), ('assistant', 'b'), ('user', 'c'), ('assistant', 'd')]
print('최근 1턴 :', keep_recent(sample, 1))

# 함정 하나 — 0턴을 넣으면 어떻게 될까요? 먼저 예상해 본 뒤 결과를 보세요.
print('0턴이면 :', keep_recent(sample, 0))

> **`0` 을 넣으면 다 지워질 것 같지만 반대로 '전부' 가 남습니다.** `-2 * 0` 은 `0` 이라 `history[0:]`, 즉 **처음부터 끝까지**가 되기 때문입니다. 음수 인덱스로 자를 때 자주 걸리는 함정이라, 0 이 들어올 수 있는 자리라면 `if n_turns <= 0: return []` 처럼 **먼저 걸러 주는** 편이 안전합니다.

**(3단계) 두 함수를 이어 써 봅니다.** 세 턴을 쌓은 뒤 최근 2턴만 남겨, 줄 수가 어떻게 변하는지 확인하세요.

In [ ]:
# 대화가 이어지는 상황을 흉내 냅니다 — 돌려받은 새 리스트를 h 에 다시 담습니다.
h = []
h = add_turn(h, '안녕하세요', '안녕하세요! 무엇을 도와드릴까요?')
h = add_turn(h, '방석 추천해줘', '메모리폼 방석을 추천드려요.')
h = add_turn(h, '세탁 되나요?', '네, 커버는 분리 세탁이 가능합니다.')

# 확인 포인트: 3턴 = 6줄, 최근 2턴만 남기면 4줄(맨 앞 인사 턴이 밀려난다).
print('전체 줄 수:', len(h))
print('최근 2턴 줄 수:', len(keep_recent(h, 2)))
for role, text in keep_recent(h, 2):
    print(' ', role, ':', text)

### 🖐️ 함께 따라하기 — 최근 1턴만 남기기

위 `keep_recent` 를 이용해 방금 만든 기록 `h` 에서 **최근 1턴(2줄)만** 남긴 리스트를 만들고, 길이가 2이며 마지막 줄의 역할이 `'assistant'` 인지 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) keep_recent(h, 1) 로 최근 1턴만 남긴 리스트를 만든다
# 2) 길이가 2인지, 마지막 줄의 역할(첫 원소)이 'assistant' 인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 대화 한 턴은 기록에서 몇 줄인가요?

<details><summary>정답 보기</summary>

**두 줄**입니다 — 사용자 질문 한 줄, 모델 답 한 줄. 그래서 최근 N턴은 뒤에서 2×N줄입니다.

</details>

**2.** `keep_recent(기록, 0)` 은 무엇을 돌려주나요?

<details><summary>정답 보기</summary>

**기록 전체**입니다. `-2 * 0` 이 `0` 이 되어 `history[0:]` — 처음부터 끝까지가 되기 때문입니다. '하나도 안 남는다' 가 아니라는 점을 기억하세요.

</details>

**3.** 누적·트리밍 함수는 왜 모델을 부르지 않고 순수 함수로 만드나요?

<details><summary>정답 보기</summary>

기록 관리는 **리스트 다루기**일 뿐 언어 모델과 무관하기 때문입니다. 순수 함수라 결과가 **항상 일정**하고 테스트·디버깅이 쉽습니다.

</details>

> **N 은 몇으로 두나요?** 정답은 없고 **토큰 예산**으로 정합니다. 기록은 매 턴 프롬프트에 통째로 들어가므로, 대화가 길어질수록 **한 번 물어보는 값이 계속 비싸집니다**(느려지기도 합니다). 그래서 실무에서는 "최근 N턴" 또는 "최근 N토큰" 으로 상한을 두고, 짧은 문의 응대는 작게(3~5턴), 맥락이 길게 이어지는 상담은 크게(10~20턴) 잡습니다. 이 교안은 7절에서 **10턴**으로 두고 씁니다.

N 을 키우는 대신 **오래된 대화를 요약해 한 줄로 접어 두는** 방법도 씁니다. 다만 잘라 내든 요약하든 **언젠가는 사라진다**는 점은 같아서, 오래 쓸 사실은 다음 절의 **장기 기억**으로 넘깁니다.

---
# 6. 장기 기억 — 잘라 낸 사실을 벡터 저장소에 남기기

## 왜 필요할까요?
5절의 트리밍은 프롬프트를 짧게 지켜 주지만, **오래된 대화를 버립니다.** 그런데 버려지는 것 중에는 "우리 강아지는 초코, 두 살 소형견"처럼 **한참 뒤에도 쓸 사실**이 섞여 있습니다. 대화 100턴 뒤에 다시 물었을 때 상담원이 초코를 모른다면 곤란하겠죠.

그래서 실무 챗봇은 기억을 **둘로 나눕니다**. 앞 절들에서 만든 것까지 합치면 셋입니다.

| | 무엇을 담나 | 어떻게 넣나 | **실무에서 쓰는 곳** |
|---|---|---|---|
| **기록 없음** | 아무것도 | 매번 새 대화 | 문서 요약·번역·분류, 상품 문구 **대량 생성**(1~3절) |
| **단기 기억** | 최근 몇 턴의 **대화 그대로** | 매번 통째로 (5절) | 고객센터 **한 세션** 상담, 주문·예약 봇 |
| **장기 기억** | 오래 쓸 **사실만** 따로 | **관련될 때만** 꺼내서 | **재방문 고객** 응대, 개인 비서, 사내 헬프데스크 |

**고르는 기준은 "세션이 끝나도 남아야 하는가" 입니다.** 상품 설명을 1,000건 만드는 일에는 기억이 아예 필요 없습니다(넣으면 토큰만 낭비). "그거 얼마예요?" 처럼 **한 상담 안에서만** 통하면 되는 것은 단기 기억으로 충분하고, 세션이 끝나면 버려도 됩니다. 하지만 "우리 아이는 닭고기 알레르기가 있어요" 는 **다음 달에 다시 와도 참인 사실**이라 어딘가에 남겨 둬야 합니다.

장기 기억은 통째로 넣을 수 없습니다 — 사실이 수백 개면 프롬프트가 다시 길어지니까요. **질문과 의미가 가까운 것만** 골라 넣어야 하는데, 그 일을 하는 도구를 우리는 이미 배웠습니다. **15~16일차의 벡터 검색**입니다. **이 절에서 새로운 것은 그 검색을 문서가 아니라 "기억"에 쓴다는 것 하나**입니다.

<img src="images/memory_short_long.png" width="820">

## 문법 — 저장과 회상
- **저장**: `기억함.add(ids=[...], documents=[사실], embeddings=[벡터])`
- **회상**: `기억함.query(query_embeddings=[질문 벡터], n_results=k)`

쓰는 도구는 15~16일차와 **똑같습니다** — 한국어 임베딩 모델과 ChromaDB. 담기는 내용만 문서 조각에서 **대화에서 알아낸 사실**로 바뀌었습니다.

**(1단계) 기억함을 준비합니다.** 아래는 15~16일차에 쓰던 그 모델·저장소라 설명 없이 제공합니다 (모델을 불러오는 데 잠시 걸립니다).

In [ ]:
# [제공 코드] 장기 기억을 담아 둘 벡터 저장소 — 15~16일차에 쓴 임베딩 모델·ChromaDB 그대로입니다.
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# output/ 는 실행 산출물 폴더입니다(저장소에 올라가지 않습니다).
OUT_DIR = Path('output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path=str(OUT_DIR / 'chroma_memory'))

# 노트북을 다시 실행해도 같은 기억이 겹쳐 쌓이지 않게, 있으면 지우고 새로 만듭니다.
if 'long_term' in [c if isinstance(c, str) else c.name for c in chroma.list_collections()]:
    chroma.delete_collection('long_term')
    
memory_box = chroma.get_or_create_collection('long_term', metadata={'hnsw:space': 'cosine'})
print('기억함 준비 완료 / 지금 담긴 사실 수:', memory_box.count())

**(2단계) 사실을 저장하는 함수를 만듭니다.** 문장을 벡터로 바꿔 기억함에 넣기만 하면 됩니다 — **모델(LLM)을 부르지 않으므로** 결과가 항상 같습니다.

In [ ]:
def remember(fact):
    """오래 쓸 사실 한 줄을 장기 기억에 저장한다."""
    # id 는 겹치면 덮어쓰기가 되므로, 지금 담긴 개수로 새 번호를 만든다.
    new_id = f'mem{memory_box.count()}'
    # 벡터 DB에 20개의 데이터가 있으면 
    # id:mem20
    
    # 벡터 DB에 0개의 데이터가 있으면 
    # id:mem0
    vector = embed_model.encode([fact], normalize_embeddings=True)
    memory_box.add(ids=[new_id], documents=[fact], embeddings=vector.tolist())
    return new_id

# 상담 중에 알게 된 사실들이라고 가정합니다.
for fact in [
    '우리 강아지 이름은 초코이고 두 살 소형견이다.',
    '초코는 닭고기 알레르기가 있어 닭고기 간식은 피해야 한다.',
    '배송은 부재 시 경비실에 맡겨 달라고 요청했다.',
]:
    remember(fact)

# 확인 포인트: 세 문장이 실제로 담겼는지.
print('담긴 사실 수:', memory_box.count())

**(3단계) 질문과 관련된 기억만 꺼내 봅니다.** 기억함에 든 세 문장 중 **무엇이 위로 올라오는지**가 핵심입니다. 여기서도 모델을 부르지 않습니다 — 검색만 하는 단계입니다.

In [ ]:
def recall(question, k=2):
    """질문과 의미가 가까운 기억 k개를 (사실, 거리) 로 돌려준다."""
    # 거리는 작을수록 가깝다 — 코사인 거리라 0 에 가까울수록 비슷한 뜻이다.
    q_vec = embed_model.encode([question], normalize_embeddings=True)
    res = memory_box.query(query_embeddings=q_vec.tolist(), n_results=k)
    return list(zip(res['documents'][0], res['distances'][0]))

# 확인 포인트: 질문이 달라지면 '위로 올라오는 기억'도 달라진다는 것.
for q in ['초코한테 줄 간식 하나 추천해줘.', '택배는 어디에 두고 가면 되나요?']:
    print('질문:', q)
    for fact, dist in recall(q):
        print(f'   거리 {dist:.3f} | {fact}')

> 같은 기억함인데 **간식 질문에는 알레르기·초코 정보**가, **배송 질문에는 경비실 요청**이 먼저 올라옵니다. 기록을 통째로 넣지 않고 **필요한 것만 골라** 넣을 수 있게 된 것입니다.

### 🖐️ 함께 따라하기 — 새 사실을 기억시키고 꺼내 보기

`remember` 로 새 사실 **"초코는 산책을 아침 일찍 나가는 것을 좋아한다."** 를 저장한 뒤, `recall` 에 질문 **"산책은 몇 시에 나가는 게 좋을까?"** 를 넣어 **1개만**(`k=1`) 꺼내 출력해 보세요.

**확인 기준**: 방금 저장한 그 문장이 1위로 나온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) remember('초코는 산책을 아침 일찍 나가는 것을 좋아한다.') 로 새 사실을 저장한다
# 2) recall('산책은 몇 시에 나가는 게 좋을까?', k=1) 로 가장 가까운 기억 하나를 꺼낸다
# 3) 그 사실과 거리를 출력한다

> **무엇을 장기 기억에 넣을 것인가** — 실무에서 품질이 갈리는 지점입니다. 기준은 하나, **"다음에 다시 만나도 참인가"** 입니다.

- 넣을 것: 알레르기·선호·배송지·구독 등급처럼 **오래 유지되는 속성**
- 넣지 말 것: "오늘 날씨 어때?" 같은 **일회성 질문**, 잡담, 이미 지난 주문 상태 — 쌓아 봐야 회상만 흐려집니다

그리고 **저장하는 순간 그 정보는 보관 대상이 됩니다.** 연락처·계좌처럼 민감한 값은 저장 전에 가리거나 아예 넣지 않는 것이 안전합니다(과제 LV2 에서 만들 마스킹 부품이 그 자리에 쓰입니다). 지우는 방법도 함께 준비해 둬야 합니다 — 손님이 "그 기억 지워 주세요" 라고 할 수 있으니까요.

---
## 이어서 — 무엇을 기억할지 모델에게 맡기기

지금까지는 **우리가 직접** `remember(...)` 를 불러 저장했습니다. 그런데 상담원이 대화 도중에 "이건 기억해 둬야지" 를 매번 판단해 손으로 넣는 것은 현실적이지 않습니다. 그래서 실무에서는 **대화 한 턴이 끝날 때마다 모델에게 한 번 더 물어봅니다** — "방금 대화에 오래 쓸 사실이 있었나?"

## 문법 — 새 문법은 없습니다
판단하는 것도 결국 **`프롬프트 | 모델 | 파서`** 체인 하나입니다. 오늘 배운 것만으로 만들 수 있습니다. 중요한 것은 **출력 형식을 좁혀 두는 것**입니다 — 사실이 있으면 **문장 하나**, 없으면 정확히 **`없음`**. 그래야 코드가 그 답을 보고 저장할지 말지 정할 수 있습니다.

<img src="images/auto_remember.png" width="820">

In [ ]:
# 대화 한 턴을 보고 '오래 쓸 사실'이 있는지 판단하는 체인 — 구조는 1~5절과 똑같다.
#   출력을 '문장 하나' 또는 '없음' 둘로 좁혀야 아래 if 문이 판단할 수 있다.
extract_prompt = ChatPromptTemplate.from_messages([
    ('system', "너는 상담 대화에서 '다음에 다시 만나도 참인 사실'만 골라 적는 정리원이다.\n"
               '- 오래 유지되는 것(알레르기·취향·사는 곳·배송 조건·예산)이 있으면 완전한 한 문장으로 적는다.\n'
               "- 지금 이 순간에만 쓰이는 것(영업시간·배송 조회·가격 문의·인사)은 정확히 '없음' 이라고만 적는다.\n"
               "- 문장 하나 또는 '없음' 외에는 아무것도 쓰지 않는다."),
    ('human', '손님: {user}\n상담원: {reply}'),
])
extract_chain = extract_prompt | model | parser

def maybe_remember(user_text, reply):
    """대화 한 턴에서 오래 쓸 사실이 나오면 장기 기억에 저장하고, 그 문장을 돌려준다."""
    fact = extract_chain.invoke({'user': user_text, 'reply': reply}).strip()
    # 모델이 '없음' 이라고 하면 저장하지 않는다 — 저장 여부를 코드가 결정한다는 점이 중요하다.
    if fact.startswith('없음'):
        return None
    remember(fact)
    return fact

print('자동 판단 준비 완료 / 지금 담긴 사실 수:', memory_box.count())

**두 종류의 대화를 넣어 봅니다.** 하나는 **오래 갈 사실**이 있고, 하나는 **지금만 쓰이는 이야기**입니다. 모델이 둘을 갈라내는지 보세요.

In [ ]:
# (손님 말, 상담원 답) 한 턴씩 — 첫 번째는 오래 갈 사실, 두 번째는 지금만 쓰이는 이야기입니다.
sample_turns = [
    ('우리 집은 3층인데 엘리베이터가 없어요.',
     '네, 배송 시 참고하겠습니다.'),

     
    ('오늘 몇 시까지 상담 되나요?',
     '오후 6시까지 가능합니다.'),
]

# 확인 포인트: 어느 쪽이 저장되고 어느 쪽이 걸러지는지.
for user_text, reply in sample_turns:
    saved = maybe_remember(user_text, reply)
    print('손님:', user_text)
    print('  →', f'저장함 — {saved}' if saved else '저장하지 않음 (없음)')

print('\n지금 담긴 사실 수:', memory_box.count())

> 배송 조건은 **다음 주문에도 참**이라 저장되고, 영업시간 문의는 **지금만 쓰이는 이야기**라 걸러졌습니다. 판단은 모델이 했지만 **저장할지 말지는 코드가 결정**합니다 — 모델은 `없음` 이라는 신호만 주고, `remember` 를 부르는 것은 우리 `if` 문입니다. 이렇게 **결정 지점을 코드에 남겨 두면** 나중에 "민감정보면 저장하지 않기" 같은 규칙을 그 자리에 더 넣을 수 있습니다.

### 🖐️ 함께 따라하기 — 자동 판단을 내 대화로 시험하기

아래 두 턴을 `maybe_remember` 에 넣어 각각 저장되는지 확인해 보세요. 하나는 **오래 갈 사실**이고 하나는 **지금만 쓰이는 이야기**입니다 — 모델이 예상대로 가르는지 보세요.

1. 손님 "초코가 닭고기를 먹으면 두드러기가 나요." / 상담원 "알겠습니다. 닭고기가 들어가지 않은 간식으로 안내드릴게요."
2. 손님 "방석 배송 언제 와요?" / 상담원 "내일 도착 예정입니다."

**확인 기준**: 1번은 저장되고, 2번은 `없음` 으로 걸러진다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 두 턴을 (손님 말, 상담원 답) 튜플의 리스트로 만든다
# 2) 각각 maybe_remember 에 넣어 결과를 출력한다
# 3) 어느 쪽이 저장되고 어느 쪽이 걸러졌는지 확인한다

In [ ]:
# 3가지 경우
# 장기기억에 넣을만한 가치가 있다고 판단한 경우 3가지 저장 방식

# 1) 기존 저장된 것과 굉장히 유사한 것이 있으면 저장을 안한다 (NOOP)
# 2) 기존 저장된 것과 유사하나 새로운 정보로 업데이트가 된 경우 --> 기존 저장된 정보를 수정한다. (UPDATE)
#   -> 우리 집은 3층에 있고 엘레베이터가 있습니다. (과거 저장)
#   -> 이사를 해사 고객의 집이 7층이고 엘레베이터가 있다. (신규 저장)
# 3) 기존에 유사한것이 없으면 저장한다. (CREATE)

# 위의 로직을 구현하기 위한 알고리즘 작성
# - extract_chain에서 저장할 가치가 있다고 판단한 경우

# 1) 저장할 fact와 기존에 저장된 정보들간의 가장 유사한 1건의 정보가져옴 
def find_nearest(fact, max_distance=0.6):
    """
    새 사실과 견줄 만큼 가까운 기억을 (id, docs, 거리)반환, 없으면 None반환
    """ 
    # 저장된 데이터 없는 경우
    if memory_box.count() == 0 :
        return None

    vector = embed_model.encode([fact], normalize_embeddings=True)
    res = memory_box.query(query_embeddings=vector.tolist(), n_results=1)
    mem_id, doc, dist = res['ids'][0][0], res['documents'][0][0], res['distances'][0][0]

    return (mem_id, doc, dist, vector.tolist()) if dist <= max_distance else None

find_nearest("초코는 알레르기가 있어요")

In [ ]:
# 2) LLM 체인으로 요청
#  2-1) 저장할 정보 fact와 유사도가 높은 정보간의 처리 방식을 요청
#  - UPDATE, ADD, NOOP에 대해서 답변을 얻음
# 3) LLM에 응답에 따라서 벡터 DB에 값을 추가, 수정, 처리안함을 동작


judge_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 기억함을 관리하는 정리원이다. 이미 저장된 사실과 새로 알게 된 사실을 견주어 한 단어로만 답한다.\n'
               '- UPDATE: 같은 대상의 같은 항목인데 내용이 달라졌다(나이·주소·취향이 바뀜). 새 사실이 옛 사실을 대체한다.\n'
               '- NOOP: 표현만 다를 뿐 이미 저장된 것과 같은 말이다.\n'
               '- ADD: 서로 다른 이야기라 둘 다 남겨야 한다.\n'
               'UPDATE, NOOP, ADD 중 한 단어 외에는 아무것도 쓰지 않는다.'),
    ('human', '저장된 사실: {old}\n새로운 사실: {new}'),
])
judge_chain = judge_prompt | model | parser

def remember_smart(fact) :
    """
    사실을 저장하되 이미 있는 기억과 겹치면 수정하거나 건너뛴다. 겹치지 않으면 저장한다.
    - (ADD|UPDATE|NOOP, fact)를 반환
    """
    near = find_nearest(fact)

    # 견줄 정보가 없으면 물어볼 것도 없이 새로 넣는다. 
    if near is None :
        remember(fact)
        return 'ADD', fact

    # 가까운 정보에 대해서 추가할건지, 수정할것인지, 건너뛸건지 물어봄
    judge = judge_chain.invoke({'old':near[1], 'new':fact}).strip().upper()
    old_id, old_doc, old_dist, new_vec = near

    if judge.startswith('NOOP') :
        # 이미 기억한 정보여서 저장하지 않고 기억한 정보를 반환
        return 'NOOP', old_doc
    elif judge.startswith('UPDATE') :
        # 기존 정보를 덮어쓰기
        memory_box.update(ids=[old_id], documents=[fact], embeddings=new_vec)
        return 'UPDATE', old_doc
    else :
        # 다른 정보여서 새로 저장
        remember(fact)
        return 'ADD', fact


def maybe_remember(user_text, reply):
    """대화 한 턴에서 오래 쓸 사실이 나오면 장기 기억에 저장하고, 그 문장을 돌려준다."""
    fact = extract_chain.invoke({'user': user_text, 'reply': reply}).strip()
    # 모델이 '없음' 이라고 하면 저장하지 않는다 — 저장 여부를 코드가 결정한다는 점이 중요하다.
    if fact.startswith('없음'):
        return None, None
    return remember_smart(fact)

print(maybe_remember('초코는 알레르기가 있어요', '알레르기가 있군요')) # 이미 저장된 정보 NOOP
print(maybe_remember('초코는 개껌을 좋합니다.', '개껌을 좋아하는 군요')) # ADD
print(maybe_remember('고객은 5층에 산다', '5층이군요')) # ADD
print(maybe_remember('고객은 이사해서 6층에 산다', '6층이군요')) # ADD

### ✅ 바로 확인 퀴즈

**1.** 단기 기억(최근 N턴)과 장기 기억은 프롬프트에 들어가는 방식이 어떻게 다른가요?

<details><summary>정답 보기</summary>

단기 기억은 최근 대화를 **통째로** 넣고, 장기 기억은 저장해 둔 사실 중 **질문과 가까운 것만 골라** 넣습니다. 그래서 기억이 아무리 많아도 프롬프트가 길어지지 않습니다.

</details>

**2.** 상품 문구를 1,000건 만드는 작업에는 어떤 기억이 필요한가요?

<details><summary>정답 보기</summary>

**아무 기억도 필요 없습니다.** 각 건이 서로 무관한 단발 작업이라 기록을 넣으면 토큰만 더 씁니다 — 1~3절처럼 체인에 `batch` 를 쓰면 됩니다. 기억은 **대화가 이어질 때** 필요한 장치입니다.

</details>

**3.** 장기 기억에서 "관련된 것만 고르는" 일은 무엇으로 하나요?

<details><summary>정답 보기</summary>

**벡터 검색**입니다(15~16일차). 사실과 질문을 각각 임베딩해 **의미가 가까운 것**을 꺼냅니다 — 낱말이 겹치지 않아도 뜻이 통하면 찾아냅니다.

</details>

**4.** 자동 판단(`maybe_remember`)에서 **저장할지 말지를 최종적으로 정하는 것**은 모델인가요, 코드인가요?

<details><summary>정답 보기</summary>

**코드입니다.** 모델은 "오래 쓸 사실 한 문장" 또는 "없음" 이라는 **신호만** 주고, 그 신호를 보고 `remember` 를 부를지 정하는 것은 우리 `if` 문입니다. 그래서 나중에 "민감정보면 저장하지 않기" 같은 규칙을 그 자리에 더 넣을 수 있습니다.

</details>

---
# 7. 종합 — 단기 기억과 장기 기억을 함께 쓰는 상담 워크플로우

이제 배운 것을 하나로 모읍니다: **역할 프롬프트 + 장기 기억 회상 + 대화 기록(`MessagesPlaceholder`) + 체인 + 누적/트리밍 함수**. 질문을 넣으면 (1) 장기 기억에서 관련된 사실을 꺼내고 (2) 최근 대화와 함께 프롬프트에 넣어 답한 뒤 (3) 그 대화를 기록에 쌓는 **상담 함수** `ask` 를 만듭니다.

**(1단계) 상담 체인을 만듭니다.** 4절의 구조에 **`{memory}` 변수 하나**를 더했습니다 — 여기에 6절에서 회상한 사실을 적어 넣습니다. 역할 → 장기 기억 → 최근 대화 → 이번 질문 순서입니다.

In [ ]:
# 역할 + 장기 기억 + 기록 자리 + 질문.
#   장기 기억은 '대화'가 아니라 '참고 사실'이라 메시지 목록이 아닌 system 안의 글로 넣는다.
counsel_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 반려동물 용품 온라인숍의 상담원이다. 존댓말로 친절하고 간결하게 답한다.\n'
               '[기억해 둔 사실]\n{memory}'),
    MessagesPlaceholder('history'),
    ('human', '{input}'),
])
counsel_chain = counsel_prompt | model | parser
print('상담 체인 준비 완료 / 프롬프트 변수:', sorted(counsel_prompt.input_variables))

**(2단계) 기록 변수와 상담 함수를 만듭니다.** 5절의 `add_turn`·`keep_recent` 와 6절의 `recall` 을 그대로 씁니다. 기록 상한은 **최근 10턴**으로 둡니다 — 대화가 아무리 길어져도 프롬프트에 들어가는 대화는 그 이상 늘지 않습니다. 아직 부르지는 않습니다 — 정의만 하고 기록이 비어 있는지 확인하세요.

In [ ]:
# 기록에 남길 최대 턴 수 — 대화가 길어져도 최근 10턴까지만 프롬프트에 들어갑니다.
#   너무 크면 프롬프트가 길어져 비용·속도에 불리하고, 너무 작으면 방금 한 말도 잊습니다.
MAX_TURNS = 10

# 최근 대화(단기 기억)를 담아 둘 변수 — 질문을 던질 때마다 여기에 쌓입니다.
chat_log = []

def ask(user_text, max_turns=MAX_TURNS):
    """장기 기억을 회상하고 최근 대화와 함께 물어본 뒤, 그 대화를 누적·트리밍한다."""
    # global : 함수 밖의 chat_log 를 이 함수가 직접 갱신하겠다는 선언
    global chat_log

    # 1) 장기 기억에서 이번 질문과 가까운 사실만 꺼내 한 덩어리 글로 만든다(거리는 버리고 문장만).
    facts = [fact for fact, _ in recall(user_text, k=3)]
    memory_text = '\n'.join(f'- {f}' for f in facts)

    # 2) 장기 기억 + 최근 대화 + 이번 질문을 함께 넣어 답을 받는다.
    answer = counsel_chain.invoke({'memory': memory_text, 'history': chat_log, 'input': user_text})
    
    # 3) 이번 대화를 누적하고 최근 max_turns 턴(기본 10턴)만 남긴다.
    chat_log = add_turn(chat_log, user_text, answer)
    chat_log = keep_recent(chat_log, max_turns)
    return answer

# 확인 포인트: 아직 한 번도 부르지 않았으므로 기록은 비어 있어야 합니다.
print('함수 준비 완료 / 지금 기록 줄 수:', len(chat_log))

**(3단계) 첫 질문을 던집니다.** 기록이 비어 있는 상태의 첫 대화입니다.

In [ ]:
print('손님  :', '우리 강아지 이름은 초코야. 소형견이고 두 살이야.')
print('상담원:', ask('우리 강아지 이름은 초코야. 소형견이고 두 살이야.'))

# 확인 포인트: 한 턴(질문+답)이 쌓여 기록이 두 줄이 됩니다.
print('기록 줄 수:', len(chat_log))

In [ ]:
chat_log

**(4단계) 이어서 묻습니다.** 이번 질문에는 '초코'가 어떤 강아지인지 적혀 있지 않습니다 — 그런데도 상담원이 알아듣는다면, 기록이 제 역할을 한 것입니다.

In [ ]:
print('손님  :', '초코한테 맞는 방석을 추천해줘.')
print('상담원:', ask('초코한테 맞는 방석을 추천해줘.'))

# 확인 포인트: 두 턴이 쌓여 네 줄. 그리고 답이 '소형견 초코'에 맞춰져 있는지 읽어 보세요.
print('기록 줄 수:', len(chat_log))

> 두 번째 질문에서 상담원이 **초코가 소형견임을 기억**하고 답했습니다. 여기까지는 **단기 기억**만으로도 되는 일입니다 — 방금 나눈 대화가 아직 `chat_log` 에 남아 있으니까요.

**(5단계) 이번엔 단기 기억만으로는 안 되는 질문을 합니다.** 아래 질문에 제대로 답하려면 **'초코는 닭고기 알레르기가 있다'** 를 알아야 하는데, 그 사실은 **이번 대화에서 나온 적이 없습니다** — 6절에서 장기 기억에 저장해 둔 것뿐입니다.

In [ ]:
# 이 질문에 대해 recall 이 무엇을 꺼내 왔는지 먼저 눈으로 봅니다.
for fact, dist in recall('초코한테 줄 간식 하나만 골라 줘.', k=5):
    print(f'   회상 | 거리 {dist:.3f} | {fact}')

print('손님  :', '초코한테 줄 간식 하나만 골라 줘.')
print('상담원:', ask('초코한테 줄 간식 하나만 골라 줘.'))

**(6단계) 이제 진짜 멀티턴으로 — `while` 로 직접 대화합니다.** 지금까지는 **우리가 셀을 하나씩 실행해** 한 턴씩 돌렸습니다. 실제 챗봇은 손님이 그만둘 때까지 **스스로** 같은 일을 반복합니다. 그 반복이 `while` 루프입니다.

루프 한 바퀴가 곧 **한 턴**입니다 — 손님 말을 받고 → `ask` 로 답하고 → 기록에 쌓기. `ask` 안에 **장기 기억 회상 + 단기 기록 누적·트리밍**이 이미 들어 있으니, 루프가 할 일은 **말을 받아 넘기고 끝내는 때를 판단**하는 것뿐입니다.

아래 셀을 실행하면 **입력창이 뜹니다.** 직접 대화해 보세요.

- **`종료`** (또는 `그만`)를 입력하면 대화가 끝납니다. 그냥 엔터만 쳐도 끝납니다.
- **`기억해: <사실>`** 형태로 입력하면 그 문장을 **장기 기억에 저장**합니다 (예: `기억해: 초코는 계단을 무서워한다`). 저장한 뒤 관련된 질문을 하면 그 사실이 회상되는지 보세요.
- 그 밖의 말은 모두 상담 질문으로 처리되어 **단기 기록**에 쌓입니다.

> `while True` 는 **스스로 멈추지 않으므로** 빠져나가는 길(`break`)을 반드시 만들어 둬야 합니다. 아래에는 두 개를 뒀습니다 — **끝내는 말**과 **최대 턴 수**. 끝내는 말을 깜빡해도 무한히 돌지 않습니다.

In [ ]:
END_WORDS = ('종료', '그만', 'exit')   # 이 말이 오면 대화를 끝낸다

turn = 0
while True:
    user_text = input('손님: ').strip()

    # 1) 끝내는 말(또는 빈 입력)이면 루프를 빠져나온다
    if user_text in END_WORDS or not user_text:
        print('상담원: 이용해 주셔서 감사합니다.')
        break

    # 2) '기억해: ...' 이면 상담이 아니라 장기 기억에 저장하고 다음 턴으로 넘어간다
    if user_text.startswith('기억해:'):
        fact = user_text.split(':', 1)[1].strip()
        remember_smart(fact)
        print('상담원: 기억해 두겠습니다 —', fact)
        continue

    # 3) 보통의 질문 — ask 가 장기 기억을 회상하고 단기 기록과 함께 물어본다
    print('상담원:', ask(user_text))
    turn += 1
    print(f'   (단기 기록 {len(chat_log)}줄 / 장기 기억 {memory_box.count()}건)')

    # 4) 지시하지 않아도 가치가 있는 정보만 자동으로 모델이 장기기억으로 저장
    saved = maybe_remember(user_text, answer)
    if saved :
        print(f" (자동 저장 - {saved})")
        
    # 5) 안전장치 — 끝내는 말을 깜빡해도 무한히 돌지 않게
    if turn >= MAX_TURNS:
        print('(최대 턴 수에 도달해 상담을 마칩니다)')
        break

> 셀 하나로 대화가 이어집니다. 우리가 새로 쓴 것은 **`while` 과 끝내는 조건**뿐이고, 기억을 다루는 일은 전부 `ask` 안에 있습니다 — 그래서 루프가 이렇게 짧습니다.

**이렇게 시험해 보세요.** 두 기억이 어떻게 다르게 동작하는지 한 번에 볼 수 있습니다.

| 입력 | 무엇을 보는가 |
|---|---|
| `기억해: 초코는 계단을 무서워한다` | **장기 기억**에 저장(건수가 하나 늘어납니다) |
| `초코가 쓸 만한 계단 용품 있어?` | 방금 저장한 사실이 **회상**되어 답에 반영되는지 |
| `그거 얼마야?` | "그거" 를 알아듣는지 — **단기 기록**이 하는 일 |
| `종료` | 루프를 빠져나가 대화가 끝나는지 |

세 번째가 특히 중요합니다. **"그거" 는 장기 기억이 아니라 단기 기록이 알아듣습니다** — 바로 앞 턴의 대화라 아직 `chat_log` 에 남아 있기 때문입니다. 반대로 `기억해:` 로 넣은 사실은 대화가 10턴을 넘겨 밀려난 뒤에도 회상됩니다. **둘은 대체재가 아니라 역할이 다릅니다.**

> **이 봇은 `기억해:` 라고 지시할 때만 장기 기억에 저장합니다.** `ask` 는 장기 기억을 **읽기만** 하고 (`recall`), 쓰지는 않습니다 — 보통의 질문은 단기 기록(`chat_log`)에만 쌓입니다. 위 표에서 두 번째·세 번째 입력 뒤에도 **장기 기억 건수가 늘지 않는 것**을 확인해 보세요.

**왜 자동으로 저장하지 않았을까요?** 무엇을 남길지는 6절에서 본 것처럼 판단이 필요한 일이고, 잘못 저장하면 **그 뒤의 모든 대화가 틀린 사실 위에서** 굴러갑니다. 게다가 한번 저장하면 그 정보는 보관 대상이 됩니다. 그래서 여기서는 **사람이 지시할 때만** 남기는 가장 단순하고 안전한 방식을 썼습니다.

**자동으로 바꾸려면 한 줄이면 됩니다.** 6절에서 만든 `maybe_remember` 를 `ask` 안에 넣으면 됩니다 — 답을 받은 뒤 `maybe_remember(user_text, answer)` 를 부르면 그 턴에서 오래 쓸 사실을 모델이 골라 저장해 줍니다. 여기서는 **턴마다 모델 호출이 한 번씩 늘기 때문에** 수동으로 두었습니다 — 실습 중 직접 바꿔 넣어 보세요.

어느 쪽을 쓰든 **모델이 엉뚱한 것을 기억하면 사람이 넣은 적도 없는 잘못된 사실이 계속 회상됩니다.** 그래서 실제 서비스는 보통 "자동 저장 + 사용자가 확인·삭제" 를 함께 둡니다.

**(7단계) 기록이 정말 10턴에서 멈추는지 확인합니다.** 대화가 길어져도 프롬프트가 무한정 길어지지 않는다는 것이 `MAX_TURNS` 를 둔 이유였죠. **모델을 부르지 않고** 순수 함수만으로 12턴을 흉내 내 확인합니다 (요금도 들지 않고 결과도 항상 같습니다).

In [ ]:
# 12턴을 쌓되, 매 턴 keep_recent 로 상한을 적용합니다 — ask 가 하는 일과 같습니다.
demo_log = []
for n in range(1, 13):
    demo_log = add_turn(demo_log, f'질문{n}', f'답{n}')
    demo_log = keep_recent(demo_log, MAX_TURNS)

# 확인 포인트: 12턴을 넣었는데도 남는 것은 10턴(20줄)뿐이고, 앞의 두 턴은 밀려났다.
print('12턴을 쌓은 뒤 남은 줄 수:', len(demo_log), f'(= {MAX_TURNS}턴 x 2줄)')
print('남아 있는 가장 오래된 질문:', demo_log[0][1])

> 12턴을 넣었지만 남은 것은 **10턴(20줄)**이고, 가장 오래된 질문이 `질문3` 입니다 — 앞의 두 턴이 밀려난 것입니다. 대화를 아무리 오래 해도 **프롬프트에 들어가는 대화 분량은 여기서 멈춥니다.**

그래서 밀려난 이야기 중 **계속 쓸 사실**은 6절처럼 **장기 기억**에 따로 남겨 두는 것입니다 — 단기 기억의 상한과 장기 기억이 짝을 이루는 이유가 이것입니다.

> 답에 **닭고기를 피하라는 이야기**가 들어 있는지 읽어 보세요. 그 사실은 최근 대화에 없었고, **장기 기억에서 회상해 프롬프트에 넣어 준 것**입니다. 이것이 트리밍으로 잘라 낸 뒤에도 오래된 사실을 쓸 수 있는 이유입니다.

이 **역할 프롬프트 + 장기 기억 회상 + `MessagesPlaceholder` 기록 + 체인** 구조는 뒤에 이어지는 **Streamlit 대시보드·챗봇** 단원에서 만드는 챗봇의 **뼈대**가 그대로 됩니다. 오늘 만든 이 위에, 다음 단원(**LangChain 에이전트·툴**)부터 한 층씩 기능을 쌓아 갑니다.

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| Runnable | 모든 부품의 공통 규약 | `invoke`·`batch`·`stream` |
| 스트리밍 | 답을 조각으로 흘려 받음 | `for chunk in chain.stream(...)` |
| RunnableLambda | 내 함수를 부품으로 | `RunnableLambda(함수)` |
| RunnableParallel | 한 입력을 동시에 여러 부품에 | `RunnableParallel(a=..., b=...)` |
| 대화 기록 | 기록을 프롬프트에 끼워 기억 부여 | `MessagesPlaceholder('history')` |
| 기록 관리 | 누적·최근 N턴 트리밍 | 순수 함수(`history[-2*n:]`) |
| 장기 기억 | 오래 쓸 사실만 저장하고 관련될 때 회상 | `memory_box.add(...)` · `memory_box.query(...)` |

- 부품이 모두 **Runnable** 이라 파이프 `|` 로 자유롭게 이어집니다.
- **기억은 마법이 아니라** 대화 기록을 프롬프트에 함께 넣는 것입니다.
- **단기**는 최근 대화를 통째로, **장기**는 저장해 둔 사실 중 **가까운 것만** 골라 넣습니다.

## ⏭️ 예고 — 다음 시간

다음 시간에는 **LangChain 에이전트·툴** 을 배웁니다. 지금까지의 체인은 **경로가 고정**되어 있었지만, 다음 단원에서는 한 걸음 더 나아간 구조로 확장합니다. 오늘 만든 대화 워크플로우가 그 바탕이 됩니다.

수고하셨습니다!